# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeerMusabih/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Note: Please ignore the first code cell. it is only to ready the notebook to connect to the github repository.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages were longer and younger than declining pages: growing content averaged about 3.2K words and 184 days old, while declining content averaged about 2.3K words and 230 days old.

The label comes from `trend_direction`, which is calculated from the change in impressions between the most recent 30 days and the previous 30 days. Pages with more than 10% growth are labeled `up`, while pages with more than 10% decline are labeled `down`.

My methodology question is whether this validation design is strong enough to support the broader interpretation. The comparison is useful as an observed portfolio-level difference, but it is observational. Age and word count may be associated with growth without causing it. The paper itself treats this finding as directional rather than causal.

### Finding 2 — The Content Performance Curve

The paper reports that content reaches its strongest health score around 61–90 days and that health declines substantially in the 271–365 day range. It also reports a rebound among 365+ day content that has been refreshed.

The label or grouping here comes from content age and freshness buckets rather than a supervised outcome label. Age is measured as days since content creation, while freshness is days since the last update.

My methodology question is whether the age and freshness comparisons can distinguish the effect of refreshing content from other differences between pages. Older pages that were refreshed may already have had stronger demand or received more attention, so the observed improvement should not be interpreted as proof that refreshing alone caused the increase. The finding is useful for prioritization, but it is best treated as directional decision-support.

## 2. My model under an honest split (before/after)

### Honest split

The Week-5 model was originally evaluated using a random split. For this audit, I re-evaluate the model using a grouped split so that pages from the same client do not appear in both the training and test sets.

This gives a more conservative estimate of how well the model may generalize to unseen clients.
### Before: Week-5 grouped validation

The Week-5 Random Forest was evaluated using a client-grouped 80/20 split. The model achieved Precision@20 of 0.70 and Precision@50 of 0.68.

This is a stronger validation design than a simple random row-level split because pages from the same client were kept together. However, it does not test whether the model generalizes forward in time.

### Temporal validation check

I checked the dataset for date and time fields before attempting a temporal validation split.

The dataset does not contain a publication date, observation date, or timestamp. The only matching field is `days_since_last_update`, which is a numeric age/freshness measure rather than an actual date.

Because there is no usable timestamp, I cannot construct a true time-based train/test split from this dataset without making assumptions that are not supported by the data.

Therefore, the grouped client split remains the main validation design. It tests generalization to unseen clients, but it does not establish how the model would perform on future data.

In [9]:
[c for c in df.columns if "date" in c.lower() or "time" in c.lower()]

['days_since_last_update']

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
TARGET = "is_declining_label"

df[TARGET] = (
    df["trend_direction"] == "down"
).astype(int)

ID_COLUMNS = [
    "content_id",
    "client_id"
]

LEAKAGE_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

NON_FEATURE_COLUMNS = [
    "provider_used",
    "model_used"
]

excluded_columns = (
    [TARGET]
    + ID_COLUMNS
    + LEAKAGE_COLUMNS
    + NON_FEATURE_COLUMNS
)

feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

print("Number of final features:", len(feature_columns))
print("\nFinal features:")
print(feature_columns)

Number of final features: 32

Final features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [11]:


audit_keywords = [
    "trend",
    "target",
    "label",
    "declin",
    "prev",
    "last_30d"
]

suspicious_features = [
    col for col in feature_columns
    if any(keyword in col.lower() for keyword in audit_keywords)
]

print("Suspicious feature names:")
print(suspicious_features)

print("\nExcluded leakage columns:")
print(LEAKAGE_COLUMNS)

print("\nExcluded identifiers:")
print(ID_COLUMNS)

Suspicious feature names:
[]

Excluded leakage columns:
['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Excluded identifiers:
['content_id', 'client_id']


### Leakage audit result

The final feature set contains 32 features and none were flagged by the keyword-based leakage check.

The target construction fields were excluded from the model:

- `trend_direction`
- `trend_pct`
- `impressions_last_30d`
- `clicks_last_30d`
- `sessions_last_30d`
- `impressions_prev_30d`
- `clicks_prev_30d`
- `sessions_prev_30d`

The identifier fields `content_id` and `client_id` were also excluded from the feature matrix. `client_id` was retained separately only for grouped validation.

The audit did not identify an obvious direct target or trend leakage field among the 32 final features. However, this is a feature-level audit rather than proof that no hidden or indirect leakage exists. The available dataset does not provide timestamps that would allow a full temporal leakage test.

## 4. Claim rewrite

### Original claim

The Random Forest provides a more useful ranking of observed declining pages than the simple rule-based score.

### Safer claim

On the held-out client-grouped test set, the Random Forest achieved higher measured Precision@20 and Precision@50 than the ML-07 rule-based baseline. This provides directional evidence that the model produced a stronger ranking of observed declining pages in this evaluation.

However, the result is decision-support evidence rather than proof of future performance or causal impact. Because the dataset does not contain a usable timestamp, this evaluation does not establish how well the model would generalize to future time periods or whether using the model to select pages for refresh would improve SEO outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.